# LaBSE Phase 1–2–3 Evaluation + Phase 3 Lambda Ablation

This notebook evaluates:

1. Original `sentence-transformers/LaBSE`
2. Phase 1: balanced all-462 Indic–Indic pairs
3. Phase 2: weak-pair weighted fine-tuning
4. Phase 3 old: weighted + distillation, `lambda=0.05`
5. Phase 3 new: weighted + stronger distillation, `lambda=0.10`

It evaluates all models on **IN22-Conv**, across all **462 directed Indic→Indic pairs**.

Run this notebook from your SSH project folder:

```text
~/labse_all_pairs_indic_finetuning
```


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Optional install cell.
# Run this only if your environment is missing packages.
# !pip install -U sentence-transformers datasets accelerate transformers huggingface_hub pandas numpy scikit-learn tqdm matplotlib

print("Optional install cell skipped.")


In [ ]:
# ============================================================
# 0. Imports and seed
# ============================================================

import os
import json
import math
import random
import hashlib
import zipfile
import gc
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

# Disable W&B prompts/noise
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 1. Project paths
# ============================================================
# SSH / VS Code: keep USE_GOOGLE_DRIVE = False and run this notebook from:
# ~/labse_all_pairs_indic_finetuning

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_all_pairs_indic_finetuning")
else:
    PROJECT_DIR = Path.cwd().resolve()

OUTPUT_DIR = PROJECT_DIR / "outputs"
METRICS_DIR = PROJECT_DIR / "metrics"
DATA_DIR = PROJECT_DIR / "data"
EXPORT_DIR = PROJECT_DIR / "exports"

EVAL_RUN_NAME = "phase123_phase3_lambda_ablation_evaluation"
EVAL_DIR = METRICS_DIR / EVAL_RUN_NAME

for d in [OUTPUT_DIR, METRICS_DIR, DATA_DIR, EXPORT_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("EXPORT_DIR:", EXPORT_DIR)


In [ ]:
# ============================================================
# 2. Model registry
# ============================================================

MODEL_SPECS = {
    # Original baseline
    "labse_baseline": {
        "type": "hf",
        "path": "sentence-transformers/LaBSE",
        "description": "Original LaBSE baseline",
    },

    # Phase 1
    "phase1_balanced_all462": {
        "type": "local",
        "path": OUTPUT_DIR / "labse_all_462_directed_pairs_balanced" / "best_model",
        "description": "Phase 1: balanced all-462 Indic-Indic pair fine-tuning",
    },

    # Phase 2
    "phase2_weak_pair_weighted": {
        "type": "local",
        "path": OUTPUT_DIR / "labse_phase2_weak_pair_weighted_from_phase1" / "best_model",
        "description": "Phase 2: weak-pair weighted fine-tuning from Phase 1",
    },

    # Old Phase 3: lambda = 0.05
    "phase3_distill_lambda005": {
        "type": "local",
        "path": OUTPUT_DIR / "labse_phase3_weighted_distillation_from_phase2" / "best_model",
        "description": "Phase 3 old: weighted + distillation, lambda=0.05",
    },

    # New Phase 3 ablation: lambda = 0.10
    "phase3_distill_lambda010": {
        "type": "local",
        "path": OUTPUT_DIR / "labse_phase3_weighted_distillation_lambda010_from_phase2" / "best_model",
        "description": "Phase 3 new: weighted + stronger distillation, lambda=0.10",
    },
}

# Validate local models
print("Models to evaluate:")
for model_name, spec in MODEL_SPECS.items():
    print("\n", model_name)
    print("  type:", spec["type"])
    print("  path:", spec["path"])
    print("  desc:", spec["description"])

    if spec["type"] == "local":
        model_path = Path(spec["path"])
        assert model_path.exists(), f"Missing model folder: {model_path}"
        assert (model_path / "modules.json").exists(), f"Invalid SentenceTransformer folder: {model_path}"

print("\nAll local model paths look valid.")


In [ ]:
# ============================================================
# 3. Evaluation configuration
# ============================================================

DATASET_NAME = "ai4bharat/IN22-Conv"
DATASET_CONFIG = "default"
DATASET_SPLIT = "test"

MAX_SEQ_LENGTH = 128
ENCODE_BATCH_SIZE = 512 if torch.cuda.is_available() else 64
COMPUTE_RETRIEVAL = True

# Keep 0 for full evaluation. Set e.g. 200 for quick debugging.
QUICK_N_ROWS = 0

# 22 scheduled Indic languages used in IN22.
LANG_CODE_MAP = {
    "asm": "asm_Beng",
    "ben": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "guj": "guj_Gujr",
    "hin": "hin_Deva",
    "kan": "kan_Knda",
    "kas": "kas_Arab",
    "gom": "gom_Deva",
    "mai": "mai_Deva",
    "mal": "mal_Mlym",
    "mni": "mni_Mtei",
    "mar": "mar_Deva",
    "npi": "npi_Deva",
    "ory": "ory_Orya",
    "pan": "pan_Guru",
    "san": "san_Deva",
    "sat": "sat_Olck",
    "snd": "snd_Deva",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
    "urd": "urd_Arab",
}

INDIC_LANGS = list(LANG_CODE_MAP.keys())
DIRECTED_PAIRS = [(src, tgt) for src in INDIC_LANGS for tgt in INDIC_LANGS if src != tgt]

print("Languages:", len(INDIC_LANGS))
print("Directed pairs:", len(DIRECTED_PAIRS))
assert len(INDIC_LANGS) == 22
assert len(DIRECTED_PAIRS) == 462

config = {
    "eval_run_name": EVAL_RUN_NAME,
    "dataset": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "dataset_split": DATASET_SPLIT,
    "num_languages": len(INDIC_LANGS),
    "num_directed_pairs": len(DIRECTED_PAIRS),
    "max_seq_length": MAX_SEQ_LENGTH,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "compute_retrieval": COMPUTE_RETRIEVAL,
    "quick_n_rows": QUICK_N_ROWS,
    "models": {k: {"type": v["type"], "path": str(v["path"]), "description": v["description"]} for k, v in MODEL_SPECS.items()},
    "seed": SEED,
}

with open(EVAL_DIR / "phase123_lambda_ablation_eval_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


In [ ]:
# ============================================================
# 4. Load IN22-Conv and resolve sentence columns
# ============================================================

def load_in22_dataset(dataset_name: str, config_name: str, split: str):
    try:
        return load_dataset(dataset_name, config_name, split=split)
    except Exception as e1:
        print("Config load failed, trying without config:", repr(e1))
        return load_dataset(dataset_name, split=split)

raw_ds = load_in22_dataset(DATASET_NAME, DATASET_CONFIG, DATASET_SPLIT)
df = pd.DataFrame(raw_ds)

print("Loaded dataset shape:", df.shape)
print("Columns:", list(df.columns))

def resolve_sentence_col(short_lang: str, data_df: pd.DataFrame) -> str:
    code = LANG_CODE_MAP[short_lang]
    candidates = [
        code,
        f"sentence_{code}",
        short_lang,
        f"sentence_{short_lang}",
    ]

    # Sindhi fallback sometimes appears as snd_Arab in some datasets.
    if short_lang == "snd":
        candidates.extend(["snd_Arab", "sentence_snd_Arab"])

    for c in candidates:
        if c in data_df.columns:
            return c

    raise KeyError(f"Could not resolve sentence column for {short_lang}. Tried: {candidates}")

SENTENCE_COLS = {lang: resolve_sentence_col(lang, df) for lang in INDIC_LANGS}

print("\nResolved sentence columns:")
for lang, col in SENTENCE_COLS.items():
    print(f"{lang:>3} -> {col}")

# Keep only rows where all 22 language columns are present and non-empty.
needed_cols = list(SENTENCE_COLS.values())
valid_mask = np.ones(len(df), dtype=bool)

for col in needed_cols:
    vals = df[col].astype(str)
    valid_mask &= df[col].notna().values
    valid_mask &= vals.str.strip().ne("").values

df_eval = df.loc[valid_mask, needed_cols].copy().reset_index(drop=True)

if QUICK_N_ROWS and QUICK_N_ROWS > 0:
    df_eval = df_eval.head(QUICK_N_ROWS).copy()

print("\nEvaluation rows:", len(df_eval))
assert len(df_eval) > 1, "Need at least two rows for random negatives/retrieval."

df_eval.to_csv(EVAL_DIR / "phase123_eval_dataset_rows_used.csv", index=False)


In [ ]:
# ============================================================
# 5. Utility functions
# ============================================================

def stable_offset(model_name: str, src_lang: str, tgt_lang: str, n: int) -> int:
    """Deterministic non-zero shift for random negative target alignment."""
    if n <= 1:
        return 0
    key = f"{model_name}|{src_lang}|{tgt_lang}|{SEED}"
    h = int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16)
    return (h % (n - 1)) + 1


def load_sentence_transformer(model_name: str, spec: dict) -> SentenceTransformer:
    path = spec["path"]
    print(f"Loading model: {model_name}")
    print("Path:", path)
    model = SentenceTransformer(str(path), device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    return model


def encode_all_languages(model: SentenceTransformer, data_df: pd.DataFrame, model_name: str) -> Dict[str, np.ndarray]:
    emb_by_lang = {}

    for lang in INDIC_LANGS:
        col = SENTENCE_COLS[lang]
        texts = data_df[col].astype(str).tolist()

        emb = model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).astype("float32")

        emb_by_lang[lang] = emb
        print(f"{model_name}: encoded {lang} | shape={emb.shape}")

    return emb_by_lang


def retrieval_metrics(src_emb: np.ndarray, tgt_emb: np.ndarray) -> dict:
    """Compute target-language retrieval metrics for aligned rows.

    Query = source sentence i.
    Candidates = all target-language sentences.
    Correct target = target sentence i.
    """
    n = src_emb.shape[0]

    if not COMPUTE_RETRIEVAL:
        return {
            "accuracy_at_1": np.nan,
            "recall_at_5": np.nan,
            "recall_at_10": np.nan,
            "mrr": np.nan,
            "mean_rank": np.nan,
            "median_rank": np.nan,
        }

    if torch.cuda.is_available():
        with torch.no_grad():
            src = torch.from_numpy(src_emb).to(DEVICE)
            tgt = torch.from_numpy(tgt_emb).to(DEVICE)

            scores = src @ tgt.T
            correct_scores = torch.diag(scores).view(-1, 1)
            ranks = (scores > correct_scores).sum(dim=1) + 1
            ranks_np = ranks.detach().cpu().numpy().astype(np.int64)

            del src, tgt, scores, correct_scores, ranks
            torch.cuda.empty_cache()
    else:
        scores = src_emb @ tgt_emb.T
        correct_scores = np.diag(scores)[:, None]
        ranks_np = (scores > correct_scores).sum(axis=1).astype(np.int64) + 1

    return {
        "accuracy_at_1": float(np.mean(ranks_np == 1)),
        "recall_at_5": float(np.mean(ranks_np <= 5)),
        "recall_at_10": float(np.mean(ranks_np <= 10)),
        "mrr": float(np.mean(1.0 / ranks_np)),
        "mean_rank": float(np.mean(ranks_np)),
        "median_rank": float(np.median(ranks_np)),
    }


def evaluate_model_by_pair(model_name: str, emb_by_lang: Dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    n = len(df_eval)

    for src_lang, tgt_lang in tqdm(DIRECTED_PAIRS, desc=f"Evaluating pairs: {model_name}"):
        src_emb = emb_by_lang[src_lang]
        tgt_emb = emb_by_lang[tgt_lang]

        gold_cos = np.sum(src_emb * tgt_emb, axis=1)

        offset = stable_offset(model_name, src_lang, tgt_lang, n)
        shifted_tgt = np.roll(tgt_emb, shift=offset, axis=0)
        random_cos = np.sum(src_emb * shifted_tgt, axis=1)

        mean_gold = float(np.mean(gold_cos))
        mean_random = float(np.mean(random_cos))
        std_gold = float(np.std(gold_cos))
        std_random = float(np.std(random_cos))
        gap = mean_gold - mean_random

        threshold = (mean_gold + mean_random) / 2.0
        sensitivity = float(np.mean(gold_cos >= threshold))
        specificity = float(np.mean(random_cos < threshold))
        balanced_accuracy = (sensitivity + specificity) / 2.0

        r_metrics = retrieval_metrics(src_emb, tgt_emb)

        rows.append({
            "model": model_name,
            "src_lang": src_lang,
            "tgt_lang": tgt_lang,
            "directed_pair": f"{src_lang}->{tgt_lang}",
            "n": n,
            "random_shift": offset,
            "mean_gold_cosine": mean_gold,
            "std_gold_cosine": std_gold,
            "mean_random_cosine": mean_random,
            "std_random_cosine": std_random,
            "cosine_gap": gap,
            "midpoint_threshold": threshold,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "balanced_accuracy": balanced_accuracy,
            **r_metrics,
        })

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 6. Run evaluation for all models
# ============================================================

all_pair_results = []

for model_name, spec in MODEL_SPECS.items():
    print("\n" + "=" * 100)
    print("Evaluating:", model_name)
    print("=" * 100)

    out_csv = EVAL_DIR / f"{model_name}_by_pair.csv"

    # Reuse if already computed.
    if out_csv.exists():
        print("Found existing result, loading:", out_csv)
        pair_df = pd.read_csv(out_csv)
        all_pair_results.append(pair_df)
        continue

    model = load_sentence_transformer(model_name, spec)
    emb_by_lang = encode_all_languages(model, df_eval, model_name)
    pair_df = evaluate_model_by_pair(model_name, emb_by_lang)

    pair_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    all_pair_results.append(pair_df)

    del model, emb_by_lang, pair_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

eval_by_pair = pd.concat(all_pair_results, ignore_index=True)
eval_by_pair.to_csv(EVAL_DIR / "phase123_eval_by_pair.csv", index=False)

print("Combined eval shape:", eval_by_pair.shape)
display(eval_by_pair.head())


In [ ]:
# ============================================================
# 7. Summary by model
# ============================================================

SUMMARY_METRICS = [
    "mean_gold_cosine",
    "std_gold_cosine",
    "mean_random_cosine",
    "std_random_cosine",
    "cosine_gap",
    "sensitivity",
    "specificity",
    "balanced_accuracy",
    "accuracy_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_rank",
    "median_rank",
]

summary_by_model = (
    eval_by_pair
    .groupby("model", as_index=False)[SUMMARY_METRICS]
    .mean()
)

model_order = list(MODEL_SPECS.keys())
summary_by_model["model"] = pd.Categorical(summary_by_model["model"], categories=model_order, ordered=True)
summary_by_model = summary_by_model.sort_values("model").reset_index(drop=True)
summary_by_model["model"] = summary_by_model["model"].astype(str)

summary_by_model.to_csv(EVAL_DIR / "phase123_eval_summary_by_model.csv", index=False)

display(summary_by_model)


In [ ]:
# ============================================================
# 8. Delta tables vs LaBSE baseline and vs Phase 1
# ============================================================

DELTA_METRICS = [
    "mean_gold_cosine",
    "mean_random_cosine",
    "cosine_gap",
    "sensitivity",
    "specificity",
    "balanced_accuracy",
    "accuracy_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_rank",
    "median_rank",
]

def make_delta_table(reference_model: str, output_name: str) -> pd.DataFrame:
    ref = eval_by_pair[eval_by_pair["model"] == reference_model].copy()
    ref = ref[["src_lang", "tgt_lang", "directed_pair"] + DELTA_METRICS]
    ref = ref.rename(columns={m: f"{m}_ref" for m in DELTA_METRICS})

    others = eval_by_pair[eval_by_pair["model"] != reference_model].copy()
    merged = others.merge(ref, on=["src_lang", "tgt_lang", "directed_pair"], how="left")

    for m in DELTA_METRICS:
        merged[f"delta_{m}"] = merged[m] - merged[f"{m}_ref"]

    out_path = EVAL_DIR / output_name
    merged.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return merged

delta_vs_labse = make_delta_table("labse_baseline", "phase123_delta_vs_labse_baseline.csv")
delta_vs_phase1 = make_delta_table("phase1_balanced_all462", "phase123_delta_vs_phase1.csv")

display(delta_vs_phase1.head())


In [ ]:
# ============================================================
# 9. Decision table vs Phase 1
# ============================================================

phase1_summary = summary_by_model[summary_by_model["model"] == "phase1_balanced_all462"].iloc[0]

decision_rows = []
for _, row in summary_by_model.iterrows():
    item = row.to_dict()
    for m in DELTA_METRICS:
        item[f"delta_{m}_vs_phase1"] = row[m] - phase1_summary[m]
    decision_rows.append(item)

decision_table = pd.DataFrame(decision_rows)

# Add a simple retrieval-first decision score.
# Higher is better. Mean rank lower is better, so subtract scaled mean-rank delta.
decision_table["decision_score_vs_phase1"] = (
    decision_table["delta_accuracy_at_1_vs_phase1"].fillna(0)
    + decision_table["delta_mrr_vs_phase1"].fillna(0)
    + decision_table["delta_recall_at_10_vs_phase1"].fillna(0)
    + decision_table["delta_specificity_vs_phase1"].fillna(0)
    + decision_table["delta_cosine_gap_vs_phase1"].fillna(0)
)

decision_table.to_csv(EVAL_DIR / "phase123_decision_table_vs_phase1.csv", index=False)

cols = [
    "model",
    "cosine_gap",
    "accuracy_at_1",
    "recall_at_10",
    "mrr",
    "specificity",
    "balanced_accuracy",
    "mean_rank",
    "delta_cosine_gap_vs_phase1",
    "delta_accuracy_at_1_vs_phase1",
    "delta_recall_at_10_vs_phase1",
    "delta_mrr_vs_phase1",
    "delta_specificity_vs_phase1",
    "decision_score_vs_phase1",
]
display(decision_table[cols].sort_values("decision_score_vs_phase1", ascending=False))


In [ ]:
# ============================================================
# 10. Weak / medium / strong bucket analysis
# ============================================================

# Define weak/strong buckets using original LaBSE baseline cosine_gap.
baseline_pairs = eval_by_pair[eval_by_pair["model"] == "labse_baseline"].copy()

q25 = baseline_pairs["cosine_gap"].quantile(0.25)
q75 = baseline_pairs["cosine_gap"].quantile(0.75)

def bucket_from_baseline_gap(x):
    if x <= q25:
        return "weak_bottom25"
    elif x >= q75:
        return "strong_top25"
    else:
        return "middle_50"

baseline_pairs["baseline_strength_bucket"] = baseline_pairs["cosine_gap"].apply(bucket_from_baseline_gap)

bucket_map = baseline_pairs[["src_lang", "tgt_lang", "directed_pair", "baseline_strength_bucket"]]
bucket_map.to_csv(EVAL_DIR / "phase123_baseline_strength_buckets.csv", index=False)

eval_with_bucket = eval_by_pair.merge(bucket_map, on=["src_lang", "tgt_lang", "directed_pair"], how="left")

weak_bucket_summary = (
    eval_with_bucket
    .groupby(["model", "baseline_strength_bucket"], as_index=False)[SUMMARY_METRICS]
    .mean()
)

weak_bucket_summary["model"] = pd.Categorical(weak_bucket_summary["model"], categories=model_order, ordered=True)
weak_bucket_summary = weak_bucket_summary.sort_values(["model", "baseline_strength_bucket"]).reset_index(drop=True)
weak_bucket_summary["model"] = weak_bucket_summary["model"].astype(str)

weak_bucket_summary.to_csv(EVAL_DIR / "phase123_weak_bucket_summary.csv", index=False)

print("LaBSE baseline cosine_gap q25:", q25)
print("LaBSE baseline cosine_gap q75:", q75)

display(weak_bucket_summary)


In [ ]:
# ============================================================
# 11. Source-language and target-language summaries
# ============================================================

source_summary = (
    eval_by_pair
    .groupby(["model", "src_lang"], as_index=False)[SUMMARY_METRICS]
    .mean()
)
source_summary["model"] = pd.Categorical(source_summary["model"], categories=model_order, ordered=True)
source_summary = source_summary.sort_values(["model", "src_lang"]).reset_index(drop=True)
source_summary["model"] = source_summary["model"].astype(str)
source_summary.to_csv(EVAL_DIR / "phase123_source_language_summary.csv", index=False)

target_summary = (
    eval_by_pair
    .groupby(["model", "tgt_lang"], as_index=False)[SUMMARY_METRICS]
    .mean()
)
target_summary["model"] = pd.Categorical(target_summary["model"], categories=model_order, ordered=True)
target_summary = target_summary.sort_values(["model", "tgt_lang"]).reset_index(drop=True)
target_summary["model"] = target_summary["model"].astype(str)
target_summary.to_csv(EVAL_DIR / "phase123_target_language_summary.csv", index=False)

print("Worst source languages by Accuracy@1 for each model:")
display(
    source_summary
    .sort_values(["model", "accuracy_at_1"])
    .groupby("model")
    .head(5)[["model", "src_lang", "accuracy_at_1", "cosine_gap", "mean_rank", "mrr"]]
)

print("Worst target languages by Accuracy@1 for each model:")
display(
    target_summary
    .sort_values(["model", "accuracy_at_1"])
    .groupby("model")
    .head(5)[["model", "tgt_lang", "accuracy_at_1", "cosine_gap", "mean_rank", "mrr"]]
)


In [ ]:
# ============================================================
# 12. Heatmaps vs Phase 1
# ============================================================

def save_delta_heatmap(delta_df: pd.DataFrame, model_name: str, metric: str):
    sub = delta_df[delta_df["model"] == model_name].copy()
    if sub.empty:
        print("No rows for", model_name)
        return

    delta_col = f"delta_{metric}"
    pivot = sub.pivot(index="src_lang", columns="tgt_lang", values=delta_col)
    pivot = pivot.reindex(index=INDIC_LANGS, columns=INDIC_LANGS)

    plt.figure(figsize=(13, 10))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label=f"{metric} delta vs Phase 1")
    plt.xticks(range(len(INDIC_LANGS)), INDIC_LANGS, rotation=90)
    plt.yticks(range(len(INDIC_LANGS)), INDIC_LANGS)
    plt.title(f"{model_name}: delta {metric} vs Phase 1")
    plt.xlabel("Target language")
    plt.ylabel("Source language")
    plt.tight_layout()

    out_path = EVAL_DIR / f"heatmap_{model_name}_delta_{metric}_vs_phase1.png"
    plt.savefig(out_path, dpi=200)
    plt.show()
    print("Saved:", out_path)

if True:
    for model_name in [
        "phase2_weak_pair_weighted",
        "phase3_distill_lambda005",
        "phase3_distill_lambda010",
    ]:
        for metric in ["cosine_gap", "accuracy_at_1"]:
            save_delta_heatmap(delta_vs_phase1, model_name, metric)


In [ ]:
# ============================================================
# 13. Top improvements and remaining weak pairs
# ============================================================

# Save top improvements/regressions vs Phase 1 for main candidate models.
for model_name in ["phase2_weak_pair_weighted", "phase3_distill_lambda005", "phase3_distill_lambda010"]:
    sub = delta_vs_phase1[delta_vs_phase1["model"] == model_name].copy()

    if sub.empty:
        continue

    top_acc = sub.sort_values("delta_accuracy_at_1", ascending=False).head(30)
    worst_acc = sub.sort_values("delta_accuracy_at_1", ascending=True).head(30)

    top_gap = sub.sort_values("delta_cosine_gap", ascending=False).head(30)
    worst_gap = sub.sort_values("delta_cosine_gap", ascending=True).head(30)

    top_acc.to_csv(EVAL_DIR / f"{model_name}_top30_accuracy_improvements_vs_phase1.csv", index=False)
    worst_acc.to_csv(EVAL_DIR / f"{model_name}_top30_accuracy_regressions_vs_phase1.csv", index=False)
    top_gap.to_csv(EVAL_DIR / f"{model_name}_top30_cosine_gap_improvements_vs_phase1.csv", index=False)
    worst_gap.to_csv(EVAL_DIR / f"{model_name}_top30_cosine_gap_regressions_vs_phase1.csv", index=False)

# Remaining weakest pairs for each model by Accuracy@1
remaining_weak_rows = []
for model_name in model_order:
    sub = eval_by_pair[eval_by_pair["model"] == model_name].copy()
    weak = sub.sort_values(["accuracy_at_1", "cosine_gap"], ascending=[True, True]).head(50)
    remaining_weak_rows.append(weak)

remaining_weak = pd.concat(remaining_weak_rows, ignore_index=True)
remaining_weak.to_csv(EVAL_DIR / "phase123_remaining_weakest_pairs_top50_each_model.csv", index=False)

display(remaining_weak.head(20))


In [ ]:
# ============================================================
# 14. Write summary text for ChatGPT upload
# ============================================================

def df_to_text_table(df, cols=None, max_rows=20):
    if cols is not None:
        df = df[cols]
    return df.head(max_rows).to_string(index=False)

summary_txt = []

summary_txt.append("# Phase 1-2-3 + Phase 3 Lambda Ablation Evaluation Summary\n")
summary_txt.append(f"Evaluation run: {EVAL_RUN_NAME}\n")
summary_txt.append(f"Dataset: {DATASET_NAME} / {DATASET_SPLIT}\n")
summary_txt.append(f"Rows: {len(df_eval)}\n")
summary_txt.append(f"Directed pairs: {len(DIRECTED_PAIRS)}\n")
summary_txt.append(f"Models: {list(MODEL_SPECS.keys())}\n")

summary_txt.append("\n## Summary by model\n")
summary_txt.append(df_to_text_table(
    summary_by_model,
    cols=["model", "cosine_gap", "accuracy_at_1", "recall_at_10", "mrr", "sensitivity", "specificity", "balanced_accuracy", "mean_rank"],
    max_rows=20,
))

summary_txt.append("\n\n## Decision table vs Phase 1\n")
summary_txt.append(df_to_text_table(
    decision_table.sort_values("decision_score_vs_phase1", ascending=False),
    cols=["model", "delta_cosine_gap_vs_phase1", "delta_accuracy_at_1_vs_phase1", "delta_recall_at_10_vs_phase1", "delta_mrr_vs_phase1", "delta_specificity_vs_phase1", "decision_score_vs_phase1"],
    max_rows=20,
))

summary_txt.append("\n\n## Weak bucket summary\n")
summary_txt.append(df_to_text_table(
    weak_bucket_summary,
    cols=["model", "baseline_strength_bucket", "cosine_gap", "accuracy_at_1", "recall_at_10", "mrr", "specificity", "mean_rank"],
    max_rows=100,
))

summary_txt.append("\n\n## Worst source languages by model\n")
summary_txt.append(df_to_text_table(
    source_summary.sort_values(["model", "accuracy_at_1"]).groupby("model").head(5),
    cols=["model", "src_lang", "accuracy_at_1", "cosine_gap", "recall_at_10", "mrr", "mean_rank"],
    max_rows=100,
))

summary_txt.append("\n")

summary_path = EVAL_DIR / "phase123_lambda_ablation_summary_for_chatgpt.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_txt))

print("Saved:", summary_path)
print("\n".join(summary_txt[:12]))


In [ ]:
# ============================================================
# 15. Export small results zip for upload back to ChatGPT
# ============================================================

zip_path = EXPORT_DIR / "phase123_phase3_lambda_ablation_results_upload_this.zip"

if zip_path.exists():
    zip_path.unlink()

keep_suffixes = {".csv", ".json", ".txt", ".png", ".jpg", ".jpeg"}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in EVAL_DIR.rglob("*"):
        if not file.is_file():
            continue
        if file.suffix.lower() in keep_suffixes:
            z.write(file, arcname=file.relative_to(EVAL_DIR))

print("Saved upload zip:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 ** 2), 2))
print("\nUpload this file back to ChatGPT:")
print(zip_path)


## What to upload back

Upload this zip file back to ChatGPT:

```text
exports/phase123_phase3_lambda_ablation_results_upload_this.zip
```

Do **not** upload model folders.
